In [10]:
from openai import OpenAI
from qdrant_client import QdrantClient
import os


In [17]:
client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

def llm(prompt):
    response = client.chat.completions.create(
        model="gemini-3.5-flash",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=1024
    )
    return response

In [3]:
embedding = OpenAI(
    api_key="lm-studio",
    base_url="http://127.0.0.1:1234/v1"
)



In [4]:
def get_embedding(text, model="text-embedding-baai-bge-m3-568m"):
    response = embedding.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding




In [5]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [6]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }

In [7]:
retrieved_context = retrieve_data(
    "What kind of earphones can I get?",
    qdrant_client,
    k=10,
)

In [8]:
def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_context"],
        context["retrieved_context_ratings"],
    ):
        formatted_context += (
            f"- ID: {id}, rating: {rating}, description: {chunk}\n"
        )

    return formatted_context


preprocessed_context = process_context(retrieved_context)

print(preprocessed_context)

- ID: B0962LKHYP, rating: 3.9, description: Delton Chroma Headphones with Microphone for Kids and Teenagers | Wired On-Ear Headphones with in-Line Mic, 3.5MM Jack, Foldable for iPhone and Android Phones, Laptops, PC, MP3 (Chroma Rose Gold) COMFORT FIT: Lightweight, flexible, foldable and adjustable. Designed for all day comfort with soft earcups and adjustable headband. These compact and portable wired headphones are the perfect everyday headset. ART+SOUND Crystal clear audio is delivered via 40mm drivers. Balanced treble and bass for accurate audio performance at an affordable price. BUILT-IN MICROPHONE: The inline mic provides high quality voice isolation for calls without the need for a bulky boom mic. Versatility for a work and play in a single headset. INLINE CONTROLS: Answer and end calls when working. Play and pause your music when relaxing or studying. Convenience for when your device is in your pocket or bag. WIDE COMPATIBILITY: 3.5mm wire jack lets you plug into most smartpho

In [9]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

In [12]:
prompt = build_prompt(
    preprocessed_context,
    "What kind of earphones can I get?"
)

print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- ID: B0962LKHYP, rating: 3.9, description: Delton Chroma Headphones with Microphone for Kids and Teenagers | Wired On-Ear Headphones with in-Line Mic, 3.5MM Jack, Foldable for iPhone and Android Phones, Laptops, PC, MP3 (Chroma Rose Gold) COMFORT FIT: Lightweight, flexible, foldable and adjustable. Designed for all day comfort with soft earcups and adjustable headband. These compact and portable wired headphones are the perfect everyday headset. ART+SOUND Crystal clear audio is delivered via 40mm drivers. Balanced treble and bass for accurate audio performance at an affordable price. BUILT-IN MICROPHONE: The inline mic provides high quality voice isolation for calls without the need for a 

In [18]:
answer = llm(prompt)

In [19]:
print(answer.choices[0].message.content)

Based on the available products, here are the different types of earphones and headphones you can get:

1. **Wired On-Ear Headphones:** 
   * **Delton Chroma Headphones (ID: B0962LKHYP):** Designed for kids and teenagers, these lightweight, foldable headphones feature soft earcups, an adjustable headband, 40mm drivers, an inline microphone, and a 3.5mm jack for wide compatibility.

2. **Wireless On-Ear Bluetooth Headphones:**
   * **Wearhaus Arc On-Ear Bluetooth Headphones (ID: B07CX3MDNY):** These feature wireless music sharing, customizable color-changing light rings, touch controls, 15 hours of playback, and noise-isolating memory foam cushions.

3. **True Wireless Earbuds:**
   * **Soul S-Gear Wireless Earbuds (ID: B08B3B57V3):** In-ear Bluetooth 5.0 earbuds with deep bass, an IPX4 sweat and water-resistant rating, up to 24 hours of total battery life (with the charging case), and an included carabiner.

4. **Two-Way Radio Earpiece/Headset:**



In [21]:
def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = llm(prompt)

    return answer